<h3>Update: 3rd July, 2019


I have played around with the PyGithub package, and it has some use ability.
I am now using the json package, and have found that, given a repo ID, it can give me a full lisy of contributors to this repo, and also find the repos owned by these contributors. Next step, create a script, which
1. picks up a repo ID from a text file
2. Makes a list of the owner and contributors to this repo.
3. Finds the license of each of these repos, if the github API lists these values.
4. Sends a pre defined Pull request containing a license text file to the owner of the repo ID which was read in step 1.
5. Repeats step 3, and concatenates each result into a single file.


In [ ]:
import sys
import pandas as pd
#!{sys.executable} -m pip install PyGithub

In [ ]:
#vital note
#ensure you pick projects from libraries.io database. This is important, since it will do away with lots of noisy repos
#plus I can then figure out which ecosystem a given samppled project sits in, and so what are the counterfactuals for our sample.


<h3>First I have codes pertaining to the Pygithub package.

In [2]:
from github import Github
import json
import requests

# First create a Github instance:

# using username and password
g = Github("swagatam11", "***password goes here***")


In [82]:
#here i will pull  a random list of 300 ID and 
#good thing: I have a way to get a random sample of 300 projects, which I then download as csv ND read into the current notebook.
#bad thing: some of the id's dont work, meaning the API call to these projects raise an exception.
#solution: get a larger csv file, say 600 rows. Then insert an exception handling line below, which drops all cases with no api reponse.
#hopefully, this will give around 300 valid rows, if not adjust size of initial csv file
#from these results I can get all the data that the API exposes. (json or csv format? also how to export?)
#Save this data and mail it to elliott 

sample=pd.read_csv('sample_df_RCT.csv')
#sample.iloc[5].project_id
repo = g.get_repo(19623908)
repo.url


'https://api.github.com/repos/kuruminbr/atom-kurumin-api'

In [83]:
#here is how you acces a repo by its ID
#repo = g.get_repo("facebook/react")#https://api.github.com/repos/facebook/react
#repo = g.get_repo('s')
repo.url

response = requests.get(repo.url)
# Display one stargazer
#print(json.dumps(response.json(), indent=0))

#the following changes the entire API response into plain text, getting rid of the json formatting. This in turn makes it easy 
#to access license, owner details etc.
todos = json.loads(response.text)
#handling a nested list
l=todos['license']#takes the owner list and assigns it to a new variable
#print(l['key'])#takes the id variable from the owner list and prints it

In [85]:
print(l)

None


In [60]:
#we will try to use json here
import json
import requests
# An unauthenticated request that doesn't contain an ?access_token=xxx query string
response = requests.get(repo.url)
# Display one stargazer
print(json.dumps(response.json(), indent=0))
#print

{
"id": 10270250,
"node_id": "MDEwOlJlcG9zaXRvcnkxMDI3MDI1MA==",
"name": "react",
"full_name": "facebook/react",
"private": false,
"owner": {
"login": "facebook",
"id": 69631,
"node_id": "MDEyOk9yZ2FuaXphdGlvbjY5NjMx",
"avatar_url": "https://avatars3.githubusercontent.com/u/69631?v=4",
"gravatar_id": "",
"url": "https://api.github.com/users/facebook",
"html_url": "https://github.com/facebook",
"followers_url": "https://api.github.com/users/facebook/followers",
"following_url": "https://api.github.com/users/facebook/following{/other_user}",
"gists_url": "https://api.github.com/users/facebook/gists{/gist_id}",
"starred_url": "https://api.github.com/users/facebook/starred{/owner}{/repo}",
"subscriptions_url": "https://api.github.com/users/facebook/subscriptions",
"organizations_url": "https://api.github.com/users/facebook/orgs",
"repos_url": "https://api.github.com/users/facebook/repos",
"events_url": "https://api.github.com/users/facebook/events{/privacy}",
"received_events_url": "https:

In [58]:
#this prints the entire API response in plain text, getting rid of the json formatting. This in turn makes it easy 
#to access license, owner details etc.
todos = json.loads(response.text)
#print(todos)
#handling a nested list
l=todos['license']#takes the owner list and assigns it to a new variable
print(l['key'])#takes the id variable from the owner list and prints it

mit


In [37]:
print(json.dumps(response.json()[2], indent=2))

KeyError: 2

In [19]:
for (k,v) in response.headers.items():
    print (k, "=>", v)


Date => Fri, 07 Jun 2019 09:35:27 GMT
Content-Type => application/json; charset=utf-8
Transfer-Encoding => chunked
Server => GitHub.com
Status => 200 OK
X-RateLimit-Limit => 60
X-RateLimit-Remaining => 59
X-RateLimit-Reset => 1559903727
Cache-Control => public, max-age=60, s-maxage=60
Vary => Accept, Accept-Encoding
ETag => W/"617d3bb0b74d66dcf004df5c5bdba858"
X-GitHub-Media-Type => github.v3; format=json
Link => <https://api.github.com/repositories/1040700/stargazers?page=2>; rel="next", <https://api.github.com/repositories/1040700/stargazers?page=40>; rel="last"
Access-Control-Expose-Headers => ETag, Link, Location, Retry-After, X-GitHub-OTP, X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset, X-OAuth-Scopes, X-Accepted-OAuth-Scopes, X-Poll-Interval, X-GitHub-Media-Type
Access-Control-Allow-Origin => *
Strict-Transport-Security => max-age=31536000; includeSubdomains; preload
X-Frame-Options => deny
X-Content-Type-Options => nosniff
X-XSS-Protection => 1; mode=block
Referrer-

In [ ]:
herer are some useful links
docs
https://buildmedia.readthedocs.org/media/pdf/pygithub/stable/pygithub.pdf

https://api.github.com/repos/facebook/react


<h3> Now I start working on JSON package, and treat github api responses as pure json objects. this is tougher, but also more powerful wrt what I can achieve. The code below is self contained, menaing I can take these lines into a diff notebook and it should run perfectly well there.

In [2]:
import urllib.parse
import requests
from github import Github
import json


# First create a Github instance, using username and password
g = Github("swagatam11", "Arthashastra@123")

In [4]:
repo = g.get_repo(19623908)
url=repo.url

In [8]:
json_data=requests.get(url).json()
print(json_data)


{'id': 19623908, 'node_id': 'MDEwOlJlcG9zaXRvcnkxOTYyMzkwOA==', 'name': 'atom-kurumin-api', 'full_name': 'kuruminbr/atom-kurumin-api', 'private': False, 'owner': {'login': 'kuruminbr', 'id': 7537511, 'node_id': 'MDQ6VXNlcjc1Mzc1MTE=', 'avatar_url': 'https://avatars2.githubusercontent.com/u/7537511?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/kuruminbr', 'html_url': 'https://github.com/kuruminbr', 'followers_url': 'https://api.github.com/users/kuruminbr/followers', 'following_url': 'https://api.github.com/users/kuruminbr/following{/other_user}', 'gists_url': 'https://api.github.com/users/kuruminbr/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/kuruminbr/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/kuruminbr/subscriptions', 'organizations_url': 'https://api.github.com/users/kuruminbr/orgs', 'repos_url': 'https://api.github.com/users/kuruminbr/repos', 'events_url': 'https://api.github.com/users/kuruminbr/events{/privacy}', 'r

In [20]:

#this gives me any field I want
x=json_data['owner']

#If the field I received is a json object itself (nested json structure), then the following format digs down 1 level.
#this is the case for the owner, and now we will get the other repos of this owner
owner_repo_url=(x['repos_url'])
#the above address gives us a fully detailed list of all repos owned by this user.


[{'login': 'zpao', 'id': 8445, 'node_id': 'MDQ6VXNlcjg0NDU=', 'avatar_url': 'https://avatars1.githubusercontent.com/u/8445?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/zpao', 'html_url': 'https://github.com/zpao', 'followers_url': 'https://api.github.com/users/zpao/followers', 'following_url': 'https://api.github.com/users/zpao/following{/other_user}', 'gists_url': 'https://api.github.com/users/zpao/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/zpao/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/zpao/subscriptions', 'organizations_url': 'https://api.github.com/users/zpao/orgs', 'repos_url': 'https://api.github.com/users/zpao/repos', 'events_url': 'https://api.github.com/users/zpao/events{/privacy}', 'received_events_url': 'https://api.github.com/users/zpao/received_events', 'type': 'User', 'site_admin': False, 'contributions': 1776}, {'login': 'sophiebits', 'id': 6820, 'node_id': 'MDQ6VXNlcjY4MjA=', 'avatar_url': 'https://a

In [38]:
#Next qn, how do i find a list of contributors to a given repo?
contrib_url='https://api.github.com/repos/facebook/react/contributors'
json_data_2=requests.get(contrib_url).json()
#the following letsme dig into each contributor's individual character, and pick out what I want.
#NB. I still dont know how to cycle through the whole list of contributors, automatically from a given contributor URL
#The following line and format allows me to dig into a given contributor's individual feature. 
print(json_data_2[4]['login'])#Here we pick the login name at contributor index 4.


acdlite


In [39]:
c1_url=(json_data_2[4]['url'])#now we pick up the address of one given contrubitor.
#We follow that URL to figure out what repositories this user owns
json_data_3=requests.get(c1_url).json()
print(json_data_3)

{'login': 'acdlite', 'id': 3624098, 'node_id': 'MDQ6VXNlcjM2MjQwOTg=', 'avatar_url': 'https://avatars0.githubusercontent.com/u/3624098?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/acdlite', 'html_url': 'https://github.com/acdlite', 'followers_url': 'https://api.github.com/users/acdlite/followers', 'following_url': 'https://api.github.com/users/acdlite/following{/other_user}', 'gists_url': 'https://api.github.com/users/acdlite/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/acdlite/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/acdlite/subscriptions', 'organizations_url': 'https://api.github.com/users/acdlite/orgs', 'repos_url': 'https://api.github.com/users/acdlite/repos', 'events_url': 'https://api.github.com/users/acdlite/events{/privacy}', 'received_events_url': 'https://api.github.com/users/acdlite/received_events', 'type': 'User', 'site_admin': False, 'name': 'Andrew Clark', 'company': '@facebook', 'blog': '', 'location'

In [ ]:
#find out, who are these contributors? I need a solid defn

In [ ]:
#experimenting with pull requests


When the syntax says
POST /repos/:owner/:repo/pulls

what it means is that the URL I feed into my program should looks like
x=requests.post('https://api.github.com/repos/swagatam11/Robot-Judge_PS/pulls') (note that first I write "repos", then name of owner, then name of repo, then the name of the end point pulls). Thisnis how you translate the syntax into python code.


In [67]:

my_url='https://api.github.com/repos/swagatam11/zurich1/pulls'

x={
    "title":"test",
    "body":"pull this in",
    "head":"zurich1/master",
    "base":"swagatam11/master"
}

token = ***
headers = {"Authorization": "token %s" % token, 'Content-Type': 'application/json'}

#"base\":\"ericdanan:testbranch\",\"head\":\"origin/master\"}", 


r = requests.post(my_url, headers=headers, data=json.dumps(x), verify=False)
#r = requests.post(url = my_url, data = x) 
print(r)

C:\APPS\Anaconda3\lib\site-packages\urllib3\connectionpool.py:847: InsecureRequestWarning: Unverified HTTPS request is being made. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#ssl-warnings
  InsecureRequestWarning)


<Response [422]>
